# Create and inspect a video experiment

Run the numbered steps from top to bottom. **Step 3 saves to the website; step 5 downloads files.**

This demo: **5 subjects × 24 trials**. Each gets **16 full videos + 4 first halves + 4 later second-half foils**, balanced **12 cut / 12 no-cut**. No videos are shared between subjects.

Start the updated local website first. It needs the measured complementary-half candidates ([server requirements](../README.md#notebook)).
Use new rehearsal subjects: earlier exposure and duplicate content are not checked.


## 1. Sign in

Edit the URL and username below. The **only prompt is your website password**.
After updating this package, choose **Kernel → Restart Kernel**, then run from this step.


In [ ]:
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Video as VideoPlayer
from dbp_api import Client, Assignments

if not hasattr(Assignments, "summary"):
    raise RuntimeError(
        "This kernel has an older DBP API loaded. Choose Kernel > Restart Kernel, "
        "then run the notebook from step 1. If this persists, launch Jupyter with "
        "'uv run --extra demo jupyter lab' from this repository."
    )

website_url = "http://127.0.0.1:8773"
username = "logben"

client = Client(website_url, timeout=120)
print(f"Signing in as {username} at {website_url}")
client.login(username, getpass("Website password: "))
print("Signed in")


## 2. See available metrics

“Measured” includes zero and false. “Unknown” means no usable measurement.


In [ ]:
inventory = client.metrics()["metrics"]
dataset = client.query_media(limit=1)
print(f"{dataset['total']:,} videos in the dataset")

coverage = []
for metric in inventory:
    values = dataset["distributions"].get(metric["id"])
    coverage.append({
        "Metric": metric["label"],
        "ID": metric["id"],
        "Measured": values["known"] if values else None,
        "Unknown": values["unknown"] if values else None,
    })
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame(coverage))


## 3. Create or open an experiment

Run the list cell first. Then set `experiment_id` to:
- **"new"** to create the 5-subject demo.
- **An ID from the table** to reopen it without changing assignments.

Leave the demo's filters empty. The seed fixes the selection when the dataset, measurements, and allocator are unchanged.


In [ ]:
display(pd.DataFrame(client.experiments(limit=100)["experiments"]))


In [ ]:
experiment_id = "73587f51122bf03e633351177d5e052c"  # Enter "new" or an existing experiment ID.

if not experiment_id:
    raise ValueError('Set experiment_id to "new" or copy an ID from the table above.')

if experiment_id == "new":
    experiment = client.create_experiment(
        name="Complementary halves v3 — 5 subjects × 24 trials",
        seed="cut-demo-24-v3",
        filters=[],
    )
    assignments = experiment.assign(
        subjects=5, items_per_subject=20,
        foils_per_block=4, balance_cuts=True,
    )
    experiment_id = assignments.experiment_id
else:
    assignments = client.assignments(experiment_id)

print("Experiment ID:", experiment_id)
print("Subjects:", ", ".join(assignments.subject_ids))


## 4. Check the assignments

Expect 24 trials, 16 full videos, 4 initial segments, 4 foils, and 12/12 cut balance for every subject.
These cut labels are automatic scene-change detections, not manual ratings.


In [ ]:
summary = pd.DataFrame(assignments.summary())
display(summary)

assert len(summary) == 5
for column, expected in {
    "trials": 24, "unique_videos": 20, "full_videos": 16,
    "initial_segments": 4, "foils": 4, "repeats": 0,
    "cut": 12, "no_cut": 12, "cuts_unknown": 0,
}.items():
    assert summary[column].eq(expected).all(), f"Unexpected {column}"


Check that each first half precedes its matching second half, with no full presentation of that parent.


In [ ]:
publication = client.experiment(experiment_id)["publication"]
assert publication["foil_policy"] == "complementary-halves-v1"
all_parents = set()

for subject in publication["subjects"]:
    trials = [trial for block in subject["blocks"] for trial in block["trials"]]
    parents = {trial["media_id"] for trial in trials}
    assert all_parents.isdisjoint(parents)
    all_parents.update(parents)
    for foil_index, foil in enumerate(trials):
        if foil["role"] != "foil":
            continue
        initial = [trial for trial in trials[:foil_index]
                   if trial["media_id"] == foil["media_id"] and trial["role"] == "parent"]
        assert len(initial) == 1
        midpoint = foil["segment"]["start_seconds"]
        assert midpoint > 0
        assert initial[0]["segment"] == {"start_seconds": 0, "end_seconds": midpoint}
        assert foil["segment"]["end_seconds"] == 2 * midpoint
        assert not any(trial["media_id"] == foil["media_id"] and trial["segment"] is None for trial in trials)

assert len(all_parents) == 100
print("Verified: 100 parent videos, 120 presentations, and valid half pairs.")


## 5. Download one subject

Choose a subject below. Run once: the download folder must be new.
The files are already trimmed—**do not crop them again**. This does not record trial completion.


In [ ]:
subject_id = "subject-001"
session = assignments.subject(subject_id, workspace=".dbp")
destination = Path("dbp-demo-download") / experiment_id / subject_id

trials = session.download(destination)
print("Downloaded to:", destination.resolve())
display(pd.DataFrame([{
    "Trial": trial.trial_id,
    "Video": trial.media_id,
    "Presentation": "second half (foil)" if trial.role == "foil" else "first half" if trial.segment else "full video",
    "Source interval": trial.segment,
    "File": str(trial.local_path),
} for trial in trials]))


## 6. Preview a pair

Play the first half, then the second half. Previews do not record participant progress.


In [ ]:
foil = next(trial for trial in trials if trial.role == "foil")
first = next(trial for trial in trials if trial.media_id == foil.media_id and trial.role == "parent")

print("First half:", first.segment)
display(VideoPlayer(filename=str(first.local_path), embed=True))
print("Second half (foil):", foil.segment)
display(VideoPlayer(filename=str(foil.local_path), embed=True))


## 7. Report playback from your task (reference only)

Your playback program—not this notebook—calls these methods.
They save progress locally and sync it to the website. Only report completion after successful playback.
Started-but-unfinished trials require review; they are not automatically replayed.


In [ ]:
# for trial in session.pending_trials:
#     session.started(trial)
#     present_video(trial.local_path)  # Your playback function.
#     session.completed(trial)
# session.sync()  # Retry uploads after a connection failure.

# Future image support would use present_image() instead.
# The current server supports videos only.


## 8. Sign out

Clear notebook outputs before sharing: previews embed video data.


In [ ]:
client.logout()
print("Signed out")
